# H1 — Reproducción en 1 clic: Economía de delegación y frontera de inversión

**Peace The Frontier Research Portfolio · Carlos Nicolás Soto**  
Working paper: `PTF-WP 3 v0.2.0` (Escenario simulado — no datos observados)  
Repositorio: `https://github.com/aisona-lab/peacethefrontier-research`

Este notebook descarga el dataset sintético publicado (`dataset.csv`, 3.300 filas), verifica su suma criptográfica `SHA-256` contra `manifest.json`, reproduce las tablas del documento y dibuja el mapa de calor del espacio de parámetros marcando dónde se invierte la conclusión entre copiloto y agente autónomo.

In [ ]:
import hashlib
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_RAW = (
    "https://raw.githubusercontent.com/aisona-lab/peacethefrontier-research/main/"
    "public/datos/experimentos/h1-delegacion-autonoma"
)

manifest_bytes = urllib.request.urlopen(f"{BASE_RAW}/manifest.json").read()
csv_bytes = urllib.request.urlopen(f"{BASE_RAW}/dataset.csv").read()
manifest = json.loads(manifest_bytes.decode("utf-8"))

expected_sha = next(f["sha256"] for f in manifest["files"] if f["role"] == "dataset")
actual_sha = hashlib.sha256(csv_bytes).hexdigest()
assert actual_sha == expected_sha, f"SHA-256 mismatch: {actual_sha} != {expected_sha}"
print(f"SHA-256 verificado OK: {actual_sha}")

df = pd.read_csv(f"{BASE_RAW}/dataset.csv")
print(f"Filas cargadas: {len(df)} | Semillas: {df['seed'].nunique()} | synthetic={df['synthetic'].all()}")
df.head(5)

In [ ]:
# Reproducción exacta de los años medianos de cruce y adopción 2031 por sector y régimen
rows = []
for (regime, sector), grp in df.groupby(["regime", "sector"], sort=False):
    crosses = []
    for seed, sgrp in grp.groupby("seed"):
        dom = sgrp[sgrp["autonomous_dominates"]]
        if not dom.empty:
            crosses.append(int(dom["year"].min()))
    crosses.sort()
    n_seeds = grp["seed"].nunique()
    median_year = crosses[(n_seeds - 1) // 2] if len(crosses) >= (n_seeds + 1) // 2 else None
    adopt_2031 = grp[grp["year"] == 2031]["autonomous_dominates"].mean()
    rows.append({
        "regime": regime,
        "sector": sector,
        "median_crossover_year": median_year,
        "adoption_2031": round(float(adopt_2031), 3),
    })

summary_df = pd.DataFrame(rows)
summary_df

In [ ]:
# Modifica estos supuestos y ejecuta la celda para ver dónde se invierte la conclusión
SECTOR = "public-administration"  # computational-science | education-tutoring | public-administration | health-workflow-admin | energy-grid-dispatch
REGIME = "baseline-coevolution"   # conservative | baseline-coevolution | verifier-infrastructure
AUDIT_COST = 3.4
BASE_DETECTION = 0.87

SECTORS = {
    "computational-science": dict(human=10, speedup=0.42, review=1.6, rec=6, liab=45, cap=0.025),
    "education-tutoring": dict(human=7, speedup=0.38, review=1.3, rec=4, liab=32, cap=0.032),
    "public-administration": dict(human=8, speedup=0.35, review=1.8, rec=7, liab=65, cap=0.022),
    "health-workflow-admin": dict(human=11, speedup=0.32, review=2.4, rec=14, liab=210, cap=0.009),
    "energy-grid-dispatch": dict(human=12, speedup=0.30, review=2.6, rec=18, liab=280, cap=0.0055),
}
REGIMES = {
    "conservative": (4.2, None),
    "baseline-coevolution": (2.4, 6.0),
    "verifier-infrastructure": (2.4, 2.0),
}

def crossover_year(p0, steps, audit_cost=AUDIT_COST, base_det=BASE_DETECTION):
    s = SECTORS[SECTOR]
    err_h, det_h = REGIMES[REGIME]
    copilot = steps * ((1 - s["speedup"]) * s["human"] + s["review"])
    for t in range(11):
        p = min(0.9999, 1 - (1 - p0) * (0.5 ** (t / err_h)))
        d = base_det if det_h is None else min(0.999, 1 - (1 - base_det) * (0.5 ** (t / det_h)))
        chain_fail = 1 - (p ** steps)
        tail_risk = chain_fail * (1 - d)
        agent = 1.8 * (0.82 ** t) + audit_cost + chain_fail * (d * s["rec"] + (1 - d) * s["liab"])
        if agent < copilot and tail_risk <= s["cap"]:
            return 2026 + t
    return np.nan

p_grid = np.linspace(0.85, 0.985, 28)
k_grid = np.arange(2, 21)
Z = np.array([[crossover_year(p, k) for p in p_grid] for k in k_grid], dtype=float)

fig, ax = plt.subplots(figsize=(9, 5))
cmap = plt.cm.viridis_r.copy()
cmap.set_bad(color="#b13b2a")
im = ax.imshow(Z, origin="lower", aspect="auto", cmap=cmap, vmin=2026, vmax=2036,
               extent=[p_grid[0], p_grid[-1], k_grid[0], k_grid[-1]])
ax.contour(p_grid, k_grid, np.isnan(Z).astype(float), levels=[0.5], colors=["#f4b940"], linewidths=2.5)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Año de cruce (rojo = sin cruce en 10 años)")
ax.set_xlabel("Fiabilidad inicial por paso p0 (2026)")
ax.set_ylabel("Longitud de cadena K (pasos)")
ax.set_title(f"H1 frontera de inversión — {SECTOR} ({REGIME}, audit={AUDIT_COST}, det={BASE_DETECTION})")
plt.tight_layout()
plt.show()